In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from datasets import load_dataset
import numpy as np
from tqdm import tqdm
import time
import pandas as pd
import os
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando:", DEVICE)

Usando: cpu


In [3]:
class HFDatasetWrapper(Dataset):
    def __init__(self, hf_dataset, transform):
        self.ds = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item["image"]
        label = int(item["label"])
        if self.transform:
            img = self.transform(img)
        return img, label

In [4]:
dataset = load_dataset("mirluvams/popocatepetl")
train_hf = dataset["train"]
test_hf = dataset["test"]

train_hf = train_hf.train_test_split(test_size=0.05, seed=42)["train"]
val_hf   = train_hf.train_test_split(test_size=0.05, seed=42)["test"]

In [5]:
img_size = 224  # resnet50 acepta cualquier tamaño > 32

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(img_size),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(int(img_size*1.14)),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

train_ds = HFDatasetWrapper(train_hf, train_transform)
val_ds = HFDatasetWrapper(val_hf, val_transform)
test_ds = HFDatasetWrapper(test_hf, val_transform)

batch_size = 16
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

In [7]:
num_labels = 4

model = models.resnet50(pretrained=True)   # o resnet18(pretrained=True)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_labels)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total params:", total_params, "Trainable:", trainable_params)

Total params: 23516228 Trainable: 23516228


In [8]:
epochs = 12
lr = 2e-4
weight_decay = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.cuda.amp.GradScaler()

C:\Users\mirlu\AppData\Local\Temp\ipykernel_13816\909384051.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
C:\Users\mirlu\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\amp\grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [9]:
def evaluate_on_loader(model, loader):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Eval", leave=False):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(imgs)
            preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            trues.extend(labels.cpu().numpy())
    acc = accuracy_score(trues, preds)
    f1_w = f1_score(trues, preds, average='weighted')
    f1_micro = f1_score(trues, preds, average='micro')
    f1_macro = f1_score(trues, preds, average='macro')
    return {"acc": acc, "f1": f1_w, "f1_micro": f1_micro, "f1_macro": f1_macro, "preds": preds, "trues": trues}

best_val = 0.0
history = []

for epoch in range(1, epochs+1):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
    for imgs, labels in pbar:
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * imgs.size(0)
        pbar.set_postfix(loss=loss.item())
    epoch_loss = running_loss / len(train_loader.dataset)
    scheduler.step()
    
    val_metrics = evaluate_on_loader(model, val_loader)
    history.append({"epoch": epoch, "loss": epoch_loss, **val_metrics})
    print(f"Epoch {epoch} - loss {epoch_loss:.4f} - val_acc {val_metrics['acc']:.4f} - val_f1 {val_metrics['f1']:.4f}")

    # Guardar mejor según f1 (o accuracy)
    if val_metrics["f1"] > best_val:
        best_val = val_metrics["f1"]
        torch.save(model.state_dict(), "best_resnet50_popo.pth")
        print("Mejor modelo guardado.")

Epoch 1/12:   0%|                                                                            | 0/12846 [00:00<?, ?it/s]C:\Users\mirlu\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Epoch 1/12:   0%|                                                                            | 0/12846 [00:05<?, ?it/s]


RuntimeError: DataLoader worker (pid(s) 6444, 3904, 8480, 16280) exited unexpectedly

In [10]:
model.load_state_dict(torch.load("best_resnet50_popo.pth"))
test_res = evaluate_on_loader(model, test_loader)
print("Test metrics:", test_res)
# Matriz de confusión
import seaborn as sns
import matplotlib.pyplot as plt
cm = confusion_matrix(test_res["trues"], test_res["preds"])
sns.heatmap(cm, annot=True, fmt='d'); plt.title("Confusion matrix"); plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'best_resnet50_popo.pth'